# Melanoma & Nevus Dermoscopy Dataset - Baseline Training (Image-only)

##### 1. Setup — chạy lại image pipeline

Dùng `%run` để tái sử dụng toàn bộ `04_image_pipeline.ipynb` (load data, Dataset, DataLoader) thay vì copy-paste lại — tránh trùng lặp code. Cần `04_image_pipeline.ipynb` nằm cùng thư mục (trên Colab: cùng thư mục với notebook này, ví dụ cùng `DRIVE_ROOT`).

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    pipeline_path = '/content/drive/MyDrive/Melanoma/04_image_pipeline.ipynb'
else:
    pipeline_path = '04_image_pipeline.ipynb'

%run {pipeline_path}

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

##### 2. Model — ResNet18 pretrained (fine-tune)

Baseline image-only: chỉ dùng ảnh, bỏ qua metadata. Thay layer `fc` cuối cùng (1000 class ImageNet) bằng 1 output duy nhất (binary classification, dùng với `BCEWithLogitsLoss`).

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

print(model.fc)

##### 3. Sanity check — forward pass thử 1 batch

In [ ]:
model.eval()
with torch.no_grad():
    sample_images, sample_labels, _ = next(iter(train_loader))
    sample_output = model(sample_images.to(device))

print('Output shape:', sample_output.shape)  # kỳ vọng (batch, 1)

##### 4. Loss, optimizer

`pos_weight` cho `BCEWithLogitsLoss` để bù mất cân bằng nhãn (train: 7631 nevus / 5065 melanoma).

In [ ]:
import torch.optim as optim

n_nevus = (df_train['diagnosis'] == 'nevus').sum()
n_melanoma = (df_train['diagnosis'] == 'melanoma').sum()
pos_weight = torch.tensor([n_nevus / n_melanoma], device=device)
print('pos_weight:', pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

##### 5. Hàm train 1 epoch / đánh giá

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score, confusion_matrix


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, labels, _ in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    for images, labels, _ in loader:
        images = images.to(device)
        labels_t = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels_t)
        total_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    preds = [1 if p >= threshold else 0 for p in all_probs]

    metrics = {
        'loss': total_loss / len(loader.dataset),
        'accuracy': accuracy_score(all_labels, preds),
        'sensitivity': recall_score(all_labels, preds),  # recall cho lớp melanoma (1)
        'auc': roc_auc_score(all_labels, all_probs),
    }
    return metrics, all_labels, preds, all_probs

##### 6. Training loop với early stopping

Theo dõi `val AUC` — lưu checkpoint tốt nhất, dừng sớm nếu không cải thiện sau `PATIENCE` epoch liên tiếp.

In [ ]:
import os

EPOCHS = 15
PATIENCE = 3

CHECKPOINT_PATH = os.path.join(DRIVE_ROOT, 'best_model.pt') if IN_COLAB else 'best_model.pt'

best_val_auc = 0.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics, _, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_metrics['auc'])

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} "
          f"| val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
          f"val_sensitivity={val_metrics['sensitivity']:.4f} val_auc={val_metrics['auc']:.4f}")

    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        epochs_no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'  -> Lưu checkpoint mới (val_auc={best_val_auc:.4f})')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping tại epoch {epoch} (không cải thiện sau {PATIENCE} epoch)')
            break

print('\nBest val AUC:', best_val_auc)

##### 7. Đánh giá cuối cùng trên test set (chỉ chạy 1 lần)

Load lại checkpoint tốt nhất (theo val), đánh giá trên test — chỉ chạy đúng 1 lần sau khi đã chốt model qua val, không dùng test để chọn lại hyperparameter.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

test_metrics, test_labels, test_preds, test_probs = evaluate(model, test_loader, criterion, device)

print('=== Test set metrics (threshold=0.5) ===')
for k, v in test_metrics.items():
    print(f'{k}: {v:.4f}')

print('\nConfusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):')
print(confusion_matrix(test_labels, test_preds))

=== Test set metrics (threshold=0.5) ===
loss: 0.6620
accuracy: 0.7933
sensitivity: 0.7886
auc: 0.8692

Confusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):
[[1298  332]
 [ 223  832]]


##### 8. Tune lại threshold trên val set (Youden's J)

Ngưỡng 0.5 chỉ là mặc định, không chắc tối ưu. Quét threshold trên **val set** (không phải test, tránh leakage), chọn threshold tối đa hóa Youden's J = sensitivity + specificity - 1 — cân bằng tốt hơn giữa bỏ sót melanoma (sensitivity) và cảnh báo nhầm (specificity).

In [ ]:
import numpy as np

val_metrics, val_labels, _, val_probs = evaluate(model, val_loader, criterion, device)

thresholds = np.arange(0.01, 1.0, 0.01)
best_threshold = 0.5
best_j = -1.0

for t in thresholds:
    preds = [1 if p >= t else 0 for p in val_probs]
    tn, fp, fn, tp = confusion_matrix(val_labels, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    j = sensitivity + specificity - 1

    if j > best_j:
        best_j = j
        best_threshold = t

print(f"Threshold mặc định: 0.50")
print(f"Threshold tối ưu (Youden's J trên val): {best_threshold:.2f} (J={best_j:.4f})")

Threshold mặc định: 0.50
Threshold tối ưu (Youden's J trên val): 0.44 (J=0.6137)


In [ ]:
print(type(model))


<class 'torchvision.models.resnet.ResNet'>


##### 9. Đánh giá lại test set với threshold đã tune

So sánh với kết quả threshold=0.5 ở mục 7 — cùng 1 model, chỉ đổi điểm cắt quyết định.

In [ ]:
test_metrics_tuned, test_labels_tuned, test_preds_tuned, _ = evaluate(
    model, test_loader, criterion, device, threshold=best_threshold
)

print(f'=== Test set metrics (threshold={best_threshold:.2f}, tuned trên val) ===')
for k, v in test_metrics_tuned.items():
    print(f'{k}: {v:.4f}')

print('\nConfusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):')
print(confusion_matrix(test_labels_tuned, test_preds_tuned))

=== Test set metrics (threshold=0.44, tuned trên val) ===
loss: 0.6620
accuracy: 0.7870
sensitivity: 0.8057
auc: 0.8692

Confusion matrix (hàng=thật, cột=dự đoán, 0=nevus, 1=melanoma):
[[1263  367]
 [ 205  850]]
